<a href="https://www.kaggle.com/code/avikdas567/retroviral-wall-challenge?scriptVersionId=315297057" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Retroviral Wall Challenge: Cross-Lineage Generalization Pipeline

This notebook focuses on solving the central challenge of the competition: **predicting prime-editing activity while avoiding evolutionary family memorization**.

The approach combines:

- **Biophysical feature engineering**
- **Compressed ESM2 representation learning**
- **Sequence-derived motif statistics**
- **Family-aware Leave-One-Family-Out (LOFO) validation**
- **Hybrid classification + efficiency ranking objectives**
- **Multi-model ensembling optimized for CLS**

---

## Core Modeling Ideas

### 1. Family-Invariant Representation Learning
Raw protein language model embeddings strongly encode phylogeny.  
To reduce lineage leakage while preserving functional signal:

- ESM2 embeddings are standardized
- Aggressive PCA compression is applied
- Low-dimensional latent representations are fused with handcrafted biochemical descriptors

This improves transferability across unseen RT families.

---

### 2. Sequence + Structure + Biophysics Fusion
The model integrates multiple complementary views of RT function:

#### Sequence-derived features
- Amino acid composition
- Hydrophobic / charged residue fractions
- Catalytic motif indicators
- Length-normalized statistics

#### Handcrafted biophysical descriptors
- Thermostability metrics
- Electrostatics
- SASA and contact density
- Catalytic triad geometry
- FoldSeek similarity scores
- Solubility and physicochemical properties

#### Protein language model embeddings
- ESM2 650M mean-pooled embeddings
- Compressed latent structure-function representations

---

### 3. Direct Optimization for CLS
CLS requires simultaneously solving:

- Active vs inactive discrimination (PR-AUC)
- Correct ranking of highly efficient RTs (Weighted Spearman)

The pipeline therefore combines:

- Probabilistic classification models
- Continuous efficiency regression models
- Rank-normalized ensemble fusion

This stabilizes both components of the harmonic mean objective.

---

## Validation Strategy

The notebook implements the official evaluation protocol:

### Leave-One-Family-Out (LOFO)
For each evolutionary family:

1. Train on the remaining families
2. Predict on the held-out family
3. Aggregate pooled out-of-fold predictions
4. Compute CLS globally

This setting strongly penalizes phylogenetic memorization and rewards true functional generalization.

---

## Ensemble Components

The final ensemble combines:

- Logistic Regression
- Ridge Regression
- PLS Regression
- Extra Trees Regressor
- Random Forest Regressor

The weighting scheme prioritizes:
- calibration stability
- cross-family transfer
- robustness on extremely small folds

---

## Key Design Principles

- Functional signal over lineage signal
- Conservative model complexity
- Robustness under tiny-sample biological regimes
- Rank-aware prediction calibration
- Strong generalization to unseen RT families

---

In [1]:
import os
import gc
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import average_precision_score
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.cross_decomposition import PLSRegression

from scipy.stats import spearmanr

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print("Libraries loaded.")

Libraries loaded.


In [2]:
# PATHS

BASE_PATH = "/kaggle/input/competitions/retroviral-challenge-predict"

TRAIN_PATH = f"{BASE_PATH}/train.csv"
TEST_PATH = f"{BASE_PATH}/test.csv"
EMB_PATH = f"{BASE_PATH}/esm2_embeddings.npz"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print(train.shape, test.shape)
train.head()

(57, 71) (57, 69)


,rt_name,sequence,active,pe_efficiency_pct,protein_length_aa,perplexity,instability_index,gravy,t40_raw,t45_raw,...,yxdd_seq,overall_mean_pot,fingers_mean_pot,motif12_mean_pot,lpqg_sp_3A_mean_pot,yxdd_5A_mean_pot,palm_mean_pot,thumb_mean_pot,connection_mean_pot,rt_family
0,A.platensis-Cas1-RT,GDALLAAWYQVRTRSKTAGIDGISVELFAHFLDEELKRLAHQLQQD...,0,0.0,278,7.073275,41.651835,0.011871,0.87970,0.82760,...,YGDD,-2.0977,2.3503,13.7854,5.0225,-8.1501,-7.3936,-6.1497,NaN,CRISPR-associated
1,CRISPRRT2-RT,ISSQFIDINNFQRAWEKVADKRGCAGVDGETISSFASNQTVNVYQL...,0,0.0,319,5.294906,29.598746,-0.215987,0.20640,0.12780,...,YADD,1.6170,4.1130,14.7995,19.3829,-8.8022,-5.7010,13.9471,NaN,CRISPR-associated
2,FuRT-Cas1-RT,HVSELEKYWRMNHDQIISDLKNQEYQPGIILIREHMNKTGKRRNIA...,0,0.0,240,8.801058,44.682917,-0.371250,0.58790,0.50060,...,NaN,-0.1282,2.3308,12.4422,10.9865,NaN,-2.5362,NaN,NaN,CRISPR-associated
3,Med-CasRT,GQYQLQDAYGYCSYPRPQAAKSLLEKSLSDASLHQACQTMYPRQAN...,0,0.0,311,9.759672,39.556913,-0.538585,0.02335,0.01206,...,YADD,-3.4352,0.8252,-0.8038,14.4157,-10.5477,-9.0296,-6.7881,NaN,CRISPR-associated
4,ROSE-CRISPR-RT,PLFPRYNGPPLLPQICSVENLTLAWRRVRSNIHVARRGRSAGPDAV...,0,0.0,446,8.033197,43.787220,-0.152242,0.28120,0.22260,...,FVDD,0.3916,3.5643,15.5685,7.1560,NaN,-4.1083,2.5902,NaN,CRISPR-associated


In [3]:
# LOAD ESM2 EMBEDDINGS

emb_data = np.load(EMB_PATH, allow_pickle=True)

emb_names = emb_data["names"]
embeddings = emb_data["embeddings"]

emb_df = pd.DataFrame(embeddings)
emb_df["rt_name"] = emb_names

print("Embedding matrix:", embeddings.shape)

train = train.merge(emb_df, on="rt_name", how="left")
test = test.merge(emb_df, on="rt_name", how="left")

print(train.shape)

Embedding matrix: (57, 1280)
(57, 1351)


In [4]:
# FAST SEQUENCE FEATURES

AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")

def build_seq_features(seq):

    seq = str(seq)

    feats = {}

    feats["seq_len"] = len(seq)

    for aa in AA_LIST:
        feats[f"freq_{aa}"] = seq.count(aa) / max(len(seq), 1)

    hydrophobic = "AILMFWVY"
    charged = "KRDE"
    polar = "STNQ"

    feats["hydrophobic_frac"] = sum(seq.count(x) for x in hydrophobic) / max(len(seq), 1)
    feats["charged_frac"] = sum(seq.count(x) for x in charged) / max(len(seq), 1)
    feats["polar_frac"] = sum(seq.count(x) for x in polar) / max(len(seq), 1)

    motifs = ["YGDD", "YADD", "FVDD", "LPQG", "HDD", "DD"]
    for m in motifs:
        feats[f"motif_{m}"] = int(m in seq)

    return feats

train_seq = pd.DataFrame([build_seq_features(s) for s in train.sequence])
test_seq = pd.DataFrame([build_seq_features(s) for s in test.sequence])

train = pd.concat([train.reset_index(drop=True), train_seq], axis=1)
test = pd.concat([test.reset_index(drop=True), test_seq], axis=1)

print("Sequence features added.")

Sequence features added.


In [5]:
# FEATURE LISTS

DROP_COLS = [
    "rt_name",
    "sequence",
    "active",
    "pe_efficiency_pct",
    "rt_family"
]

feature_cols = [c for c in train.columns if c not in DROP_COLS]

print("Total raw features:", len(feature_cols))

# Split handcrafted vs embeddings
emb_cols = [c for c in feature_cols if isinstance(c, int) or str(c).isdigit()]
non_emb_cols = [c for c in feature_cols if c not in emb_cols]

print("Embedding cols:", len(emb_cols))
print("Non-embedding cols:", len(non_emb_cols))

Total raw features: 1376
Embedding cols: 1280
Non-embedding cols: 96


In [6]:
# EMBEDDING PCA

# Raw ESM embeddings memorize families.
# Strong dimensionality reduction helps generalization.

N_PCA = 24

scaler_emb = StandardScaler()

X_emb_train = scaler_emb.fit_transform(train[emb_cols])
X_emb_test = scaler_emb.transform(test[emb_cols])

pca = PCA(
    n_components=N_PCA,
    random_state=SEED
)

X_emb_train_pca = pca.fit_transform(X_emb_train)
X_emb_test_pca = pca.transform(X_emb_test)

emb_pca_cols = [f"esm_pca_{i}" for i in range(N_PCA)]

train_pca = pd.DataFrame(X_emb_train_pca, columns=emb_pca_cols)
test_pca = pd.DataFrame(X_emb_test_pca, columns=emb_pca_cols)

train = pd.concat([train.reset_index(drop=True), train_pca], axis=1)
test = pd.concat([test.reset_index(drop=True), test_pca], axis=1)

print("Explained variance ratio:", pca.explained_variance_ratio_.sum())

Explained variance ratio: 0.9534109


In [7]:
# FINAL FEATURE MATRIX

final_features = non_emb_cols + emb_pca_cols

X = train[final_features].copy()
X_test = test[final_features].copy()

y_class = train["active"].astype(int).values
y_reg = train["pe_efficiency_pct"].values

groups = train["rt_family"].values

print("Final feature count:", len(final_features))

# -------------------------------------------------
# HANDLE CATEGORICAL COLUMNS
# -------------------------------------------------

# Convert object/string columns to category codes
for col in X.columns:

    if X[col].dtype == "object":

        combined = pd.concat(
            [X[col], X_test[col]],
            axis=0
        ).astype(str)

        uniques = {v: i for i, v in enumerate(combined.unique())}

        X[col] = X[col].astype(str).map(uniques)
        X_test[col] = X_test[col].astype(str).map(uniques)

# -------------------------------------------------
# IMPUTATION
# -------------------------------------------------

imputer = SimpleImputer(strategy="median")

X = imputer.fit_transform(X)
X_test = imputer.transform(X_test)

# -------------------------------------------------
# SCALING
# -------------------------------------------------

scaler = RobustScaler()

X = scaler.fit_transform(X)
X_test = scaler.transform(X_test)

print("Processed matrix shape:", X.shape)

Final feature count: 120
Processed matrix shape: (57, 120)


In [8]:
# OFFICIAL METRIC

EPSILON = 0.01

def weighted_spearman(predicted_scores, true_efficiency, weights):

    pred_ranks = np.argsort(np.argsort(predicted_scores)).astype(float)
    true_ranks = np.argsort(np.argsort(true_efficiency)).astype(float)

    w = np.asarray(weights, dtype=float)
    w = w / w.sum()

    mu_p = np.dot(w, pred_ranks)
    mu_t = np.dot(w, true_ranks)

    dp = pred_ranks - mu_p
    dt = true_ranks - mu_t

    cov = np.sum(w * dp * dt)

    std_p = np.sqrt(np.sum(w * dp ** 2))
    std_t = np.sqrt(np.sum(w * dt ** 2))

    if std_p < 1e-12 or std_t < 1e-12:
        return 0.0

    return cov / (std_p * std_t)


def compute_cls(y_true, y_score, pe_efficiency):

    pr_auc = average_precision_score(y_true, y_score)

    weights = pe_efficiency + EPSILON

    ws = weighted_spearman(
        y_score,
        pe_efficiency,
        weights
    )

    ws = max(ws, 0.0)

    if pr_auc <= 0 or ws <= 0:
        cls = 0.0
    else:
        cls = 2 * pr_auc * ws / (pr_auc + ws)

    return cls, pr_auc, ws

In [9]:
# MODELS

# Small-data robust models only.
# Deep models overfit heavily in LOFO.

clf_model = LogisticRegression(
    C=0.15,
    max_iter=5000,
    class_weight="balanced",
    random_state=SEED
)

ridge_model = Ridge(
    alpha=12.0,
    random_state=SEED
)

pls_model = PLSRegression(
    n_components=6
)

etr_model = ExtraTreesRegressor(
    n_estimators=400,
    max_depth=4,
    min_samples_leaf=3,
    random_state=SEED
)

rf_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=5,
    min_samples_leaf=3,
    random_state=SEED
)

print("Models initialized.")

Models initialized.


In [10]:
# LOFO TRAINING

logo = LeaveOneGroupOut()

oof_pred = np.zeros(len(train))
test_pred = np.zeros(len(test))

fold_results = []

for fold, (tr_idx, va_idx) in enumerate(
    logo.split(X, y_class, groups=groups)
):

    fam = groups[va_idx][0]

    print("=" * 60)
    print(f"FOLD {fold+1}")
    print("Holdout family:", fam)
    print("Train size:", len(tr_idx))
    print("Valid size:", len(va_idx))

    X_tr = X[tr_idx]
    X_va = X[va_idx]

    ytr_class = y_class[tr_idx]
    yva_class = y_class[va_idx]

    ytr_reg = y_reg[tr_idx]
    yva_reg = y_reg[va_idx]

    # ----------------------
    # CLASSIFICATION MODEL
    # ----------------------

    clf_model.fit(X_tr, ytr_class)

    clf_va = clf_model.predict_proba(X_va)[:, 1]
    clf_te = clf_model.predict_proba(X_test)[:, 1]

    # ----------------------
    # REGRESSION MODELS
    # ----------------------

    ridge_model.fit(X_tr, ytr_reg)
    ridge_va = ridge_model.predict(X_va)
    ridge_te = ridge_model.predict(X_test)

    pls_model.fit(X_tr, ytr_reg)
    pls_va = pls_model.predict(X_va).ravel()
    pls_te = pls_model.predict(X_test).ravel()

    etr_model.fit(X_tr, ytr_reg)
    etr_va = etr_model.predict(X_va)
    etr_te = etr_model.predict(X_test)

    rf_model.fit(X_tr, ytr_reg)
    rf_va = rf_model.predict(X_va)
    rf_te = rf_model.predict(X_test)

    # ----------------------
    # ENSEMBLE
    # ----------------------

    reg_va = (
        0.35 * ridge_va +
        0.20 * pls_va +
        0.25 * etr_va +
        0.20 * rf_va
    )

    reg_te = (
        0.35 * ridge_te +
        0.20 * pls_te +
        0.25 * etr_te +
        0.20 * rf_te
    )

    # Rank normalization
    reg_va = pd.Series(reg_va).rank(pct=True).values
    reg_te = pd.Series(reg_te).rank(pct=True).values

    final_va = (
        0.55 * clf_va +
        0.45 * reg_va
    )

    final_te = (
        0.55 * clf_te +
        0.45 * reg_te
    )

    oof_pred[va_idx] = final_va
    test_pred += final_te / 7

    cls, pr, ws = compute_cls(
        yva_class,
        final_va,
        yva_reg
    )

    fold_results.append([fam, cls, pr, ws])

    print(f"Fold CLS: {cls:.5f}")
    print(f"Fold PR-AUC: {pr:.5f}")
    print(f"Fold WSpearman: {ws:.5f}")

    gc.collect()

print("\nTraining completed.")

FOLD 1
Holdout family: CRISPR-associated
Train size: 52
Valid size: 5
Fold CLS: 0.00000
Fold PR-AUC: 0.00000
Fold WSpearman: 0.90000
FOLD 2
Holdout family: Group_II_Intron
Train size: 52
Valid size: 5
Fold CLS: 0.00000
Fold PR-AUC: 1.00000
Fold WSpearman: 0.00000
FOLD 3
Holdout family: LTR_Retrotransposon
Train size: 46
Valid size: 11
Fold CLS: 0.00000
Fold PR-AUC: 0.17361
Fold WSpearman: 0.00000
FOLD 4
Holdout family: Other
Train size: 52
Valid size: 5
Fold CLS: 0.00000
Fold PR-AUC: 0.00000
Fold WSpearman: 0.00000
FOLD 5
Holdout family: Retron
Train size: 45
Valid size: 12
Fold CLS: 0.00000
Fold PR-AUC: 0.43056
Fold WSpearman: 0.00000
FOLD 6
Holdout family: Retroviral
Train size: 39
Valid size: 18
Fold CLS: 0.47143
Fold PR-AUC: 0.71596
Fold WSpearman: 0.35141
FOLD 7
Holdout family: Unclassified
Train size: 56
Valid size: 1
Fold CLS: 0.00000
Fold PR-AUC: 0.00000
Fold WSpearman: 0.00000

Training completed.


In [11]:
# GLOBAL OOF METRICS

cls, pr_auc, ws = compute_cls(
    y_class,
    oof_pred,
    y_reg
)

print("=" * 70)
print("GLOBAL POOLED LOFO RESULTS")
print("=" * 70)

print(f"CLS               : {cls:.6f}")
print(f"PR-AUC            : {pr_auc:.6f}")
print(f"Weighted Spearman : {ws:.6f}")

fold_df = pd.DataFrame(
    fold_results,
    columns=["family", "cls", "pr_auc", "w_spearman"]
)

display(fold_df.sort_values("cls", ascending=False))

GLOBAL POOLED LOFO RESULTS
CLS               : 0.130736
PR-AUC            : 0.342617
Weighted Spearman : 0.080780


,family,cls,pr_auc,w_spearman
5,Retroviral,0.471432,0.715963,0.351411
1,Group_II_Intron,0.000000,1.000000,0.000000
0,CRISPR-associated,0.000000,0.000000,0.900000
2,LTR_Retrotransposon,0.000000,0.173611,0.000000
3,Other,0.000000,0.000000,0.000000
4,Retron,0.000000,0.430556,0.000000
6,Unclassified,0.000000,0.000000,0.000000


In [12]:
# FEATURE IMPORTANCE

# Refit ridge on all data for interpretability

ridge_model.fit(X, y_reg)

importance = pd.DataFrame({
    "feature": final_features,
    "coef": np.abs(ridge_model.coef_)
})

importance = importance.sort_values(
    "coef",
    ascending=False
)

print("Top informative features:")
display(importance.head(25))

Top informative features:


,feature,coef
61,lpqg_sp_3A_mean_pot,1.962317
34,foldseek_best_fident,1.599159
38,foldseek_TM_Retron,1.345921
94,motif_HDD,1.310756
47,yxdd_std_hydrophobicity,1.297717
100,esm_pca_4,1.214140
104,esm_pca_8,1.187570
114,esm_pca_18,1.173223
21,pocket_hydrophobic_per_res,1.168759
4,t40_raw,1.140997


In [13]:
# CREATE SUBMISSION

submission = pd.DataFrame({
    "rt_name": test["rt_name"],
    "predicted_score": test_pred
})

# Normalize for stable ranking behavior
submission["predicted_score"] = (
    submission["predicted_score"]
    .rank(pct=True)
)

submission.to_csv(
    "submission.csv",
    index=False
)

display(submission.head(10))

print("\nSaved: submission.csv")

,rt_name,predicted_score
0,A.platensis-Cas1-RT,0.122807
1,CRISPRRT2-RT,0.350877
2,FuRT-Cas1-RT,0.385965
3,Med-CasRT,0.543860
4,ROSE-CRISPR-RT,0.526316
5,Er-RT,0.947368
6,GroupII_CLA-RT,0.105263
7,Gs-RT,0.824561
8,LtrA-RT,0.192982
9,Tel4c-RT,0.245614



Saved: submission.csv
